# Testando busca em largura

In [ ]:
# CORREÇÃO 1: Passe os argumentos necessários para a função
def return_links_from_href(hrefs, base):
    arr = []
    for a in hrefs:
        # Garanta que não está duplicando barras
        link_completo = base + a['href']
        arr.append(link_completo)
    return arr

 # Entrar no link e pegar os links do primeiro nível.

# por na fila

# Entrar no sites do primeiro nivel

# adicionar a si mesmo como verificado

# Pegar links filhos e colocar na fila se o link já não estiver sido verificado

# Deque a si mesmo.

# Pegar dados (Somente no cenário real)

# Repetir passo 1.

# Se a fila estiver vazia encerrar algoritmo. 

In [48]:
import nest_asyncio
nest_asyncio.apply()  # Essencial para não travar o Kernel do Jupyter

from seleniumbase import SB
from collections import deque
from urllib.parse import urljoin

# Configurações de caminho e URL
# Certifique-se de que o Live Server (porta 5500) está rodando no VS Code
base_url = "http://127.0.0.1:5500/app/notbooks/site_teste/"                                                                
index = "index.html"
initial_page = base_url + index

# Inicialização de controle (fora do loop)
verificados = set()




with SB(browser='chrome', headless=True) as sb:
    sb.open(initial_page)
    
    # Função interna para pegar links da página ATUAL
    def get_links_da_pagina_atual(driver):
        soup = driver.get_beautiful_soup()
        hrefs = soup.find_all('a', href=True)
        links_completos = []
        for a in hrefs:
            # urljoin resolve o link relativo baseado na URL em que o navegador está agora
            # Ex: se está em site_teste/la_liga/ e o link é 'index.html', 
            # ele gera site_teste/la_liga/index.html corretamente.
            url_completa = urljoin(driver.get_current_url(), a['href'])
            links_completos.append(url_completa)
        return links_completos

    # Inicializa a fila
    fila_de_pesquisa = deque(get_links_da_pagina_atual(sb))
    verificados.add(initial_page)

    while fila_de_pesquisa:
        new_tab = fila_de_pesquisa.popleft()
        
        # Limpeza simples para evitar âncoras (ex: index.html#topo)
        new_tab = new_tab.split('#')[0]

        if new_tab not in verificados:
            print(f"Navegando para: {new_tab}")
            verificados.add(new_tab)
            
            try:
                sb.open(new_tab)
                # Pega os links da nova página e adiciona na fila
                novos_links = get_links_da_pagina_atual(sb)
                fila_de_pesquisa.extend(novos_links)
            except Exception as e:
                print(f"Erro ao abrir {new_tab}: {e}")
        

Navegando para: http://127.0.0.1:5500/app/notbooks/site_teste/premier_league/index.html
Navegando para: http://127.0.0.1:5500/app/notbooks/site_teste/la_liga/index.html
Navegando para: http://127.0.0.1:5500/app/notbooks/site_teste/serie_a/index.html
Navegando para: http://127.0.0.1:5500/app/notbooks/site_teste/premier_league/time_1/index.html
Navegando para: http://127.0.0.1:5500/app/notbooks/site_teste/premier_league/time_2/index.html
Navegando para: http://127.0.0.1:5500/app/notbooks/site_teste/premier_league/time_3/index.html
Navegando para: http://127.0.0.1:5500/app/notbooks/site_teste/la_liga/time_1/index.html
Navegando para: http://127.0.0.1:5500/app/notbooks/site_teste/la_liga/time_2/index.html
Navegando para: http://127.0.0.1:5500/app/notbooks/site_teste/la_liga/time_3/index.html
Navegando para: http://127.0.0.1:5500/app/notbooks/site_teste/serie_a/time_1/index.html
Navegando para: http://127.0.0.1:5500/app/notbooks/site_teste/serie_a/time_2/index.html
Navegando para: http://12

# Testando busca e adicionar em meio de persistência.

# . Entrar em link presente na fila do algoritmo BFS.
# . pegar pegar todas as tabelas, dado código deterministico para filtrar todo ruído do HTML
# . Retirar o Caption ( Titulo da Tabela )
# . Criar dicionário vazio.
# . thead => tr => Retirar os nomes das colunas em thead[tr] e inserir no dicionário. obs.: labels do dicionário variam de acordo com o tamanho do vetor tr dentro de thead.
# . tbody => tr => 
# . 
# . th => tablble header
# . td => tabble data
# . Montar um dicionário referente ao THEAD ( Nome do dicionário )
# . 
# . Inserir em uma fila
# . dequeueleft
# . Entrar em cada link ( inserir o proprio link em lista de verificados ) 
# Repetir 1.

In [ ]:
from seleniumbase import SB
import json
from bs4 import BeautifulSoup
import pandas as pd

def table_to_json(table):
    headers = [th.get_text(strip=True) for th in table.select('thead th')]
    rows = []
    
    for tr in table.select('tbody tr'):
        th = tr.find('th')
        a = th.find('a') if th else None
        row = {
            "meta": {
                "competicao": th.get_text(strip=True) if th else None,
                "link": a['href'] if a else None
            },
            "dados": {}
        }
        
        for i, td in enumerate(tr.select('td')):
            coluna = headers[i + 1] if i + 1 < len(headers) else f"col_{i}"
            td_a = td.find('a')
            row["dados"][coluna] = {
                "valor": td.get_text(strip=True),
                "link": td_a['href'] if td_a else None
            }
        
        rows.append(row)
    
    return rows

with SB(uc=True) as sb:
    url = "https://fbref.com/en/comps/"
    sb.uc_open_with_reconnect(url, 4)
    sb.uc_gui_click_captcha()
    
    resultado = sb.execute_script("""
    const tabelasReais = Array.from(document.querySelectorAll('table')).filter(table => {
        const temLinhasComDados = table.querySelectorAll('tbody tr td').length > 0;
        const estaVisivel = table.offsetWidth > 0 && table.offsetHeight > 0;
        return temLinhasComDados && estaVisivel;
    });
    return tabelasReais.map(table => table.outerHTML);
""")
    tabbles = list(map(lambda x: BeautifulSoup(x,'html.parser'), resultado ))

jsons = list(map(lambda x: table_to_json(x), tabbles))



In [22]:
jsons

[[{'meta': {'competicao': 'FIFA Club World Cup',
    'link': '/en/comps/719/history/FIFA-Club-World-Cup-Seasons'},
   'dados': {'Gender': {'valor': 'M', 'link': None},
    'Governing Body': {'valor': 'FIFA', 'link': None},
    'First Season': {'valor': '2019',
     'link': '/en/comps/719/2019/2019-FIFA-Club-World-Cup-Stats'},
    'Last Season': {'valor': '2025',
     'link': '/en/comps/719/FIFA-Club-World-Cup-Stats'},
    'Tier': {'valor': '', 'link': None},
    'Awards': {'valor': '', 'link': None}}},
  {'meta': {'competicao': 'CONCACAF Champions Cup',
    'link': '/en/comps/133/history/CONCACAF-Champions-Cup-Seasons'},
   'dados': {'Gender': {'valor': 'M', 'link': None},
    'Governing Body': {'valor': 'CONCACAF', 'link': None},
    'First Season': {'valor': '2020',
     'link': '/en/comps/133/2020/2020-CONCACAF-Champions-Cup-Stats'},
    'Last Season': {'valor': '2026',
     'link': '/en/comps/133/CONCACAF-Champions-Cup-Stats'},
    'Tier': {'valor': '1st', 'link': None},
    'Award